In [1]:
import os
os.chdir("/Users/nautilus/cobblestone_study")

import pandas as pd
from src.features import build_features, FEATURE_COLS
from src.models import train_lgbm, predict_lgbm
from src.curve import baseload_peak, signal_to_noise
from config import CLEAN_PARQUET, PEAK_HOURS, PEAK_WEEKDAYS
import json

df = pd.read_parquet(CLEAN_PARQUET)
df = build_features(df)

print(df.shape)
print(df.columns.tolist())


(17181, 17)
['prices', 'load_fc', 'wind_fc', 'solar_fc', 'actual_load', 'actual_gen', 'residual_load_fc', 'hour', 'dayofweek', 'month', 'is_holiday', 'is_peak', 'price_lag_24', 'price_lag_168', 'price_roll_mean_24', 'price_roll_std_24', 'price_roll_mean_168']


In [3]:
TEST_WEEKS = 10
WEEK_HOURS = 7 * 24
REFERENCE_WEEKS = 4

with open("outputs/metrics.json") as f:
    metrics = json.load(f)
mae = metrics["lgbm"]["mae"]
rmse = metrics["lgbm"]["rmse"]

print(f"Test period: {TEST_WEEKS} weeks = {TEST_WEEKS * WEEK_HOURS} hours")
print(f"Model MAE: {mae}, RMSE: {rmse}")
print(f"Data ends at: {df.index[-1]}")


Test period: 10 weeks = 1680 hours
Model MAE: 16.66, RMSE: 28.58
Data ends at: 2026-06-11 23:00:00+02:00


In [5]:
results = []

for week in range(TEST_WEEKS, 0, -1):
    cutoff = len(df) - week * WEEK_HOURS
    
    train_df = df.iloc[:cutoff]
    test_df = df.iloc[cutoff : cutoff + WEEK_HOURS]
    
    # train and forecast
    model = train_lgbm(train_df[FEATURE_COLS], train_df["prices"])
    forecast = predict_lgbm(model, test_df[FEATURE_COLS])
    
    fc_base, fc_peak = baseload_peak(forecast)
    
    # Reference : trailing 4 weeks of realised prices BEFORE the cutoff
    ref_window = train_df["prices"].iloc[-(REFERENCE_WEEKS * WEEK_HOURS):]
    ref_base, ref_peak = baseload_peak(ref_window)
    
    # Realised prices this week
    real_base, real_peak = baseload_peak(test_df["prices"])
    
   # Signal
    spread = fc_base - ref_base
    signal = "LONG" if spread > 0 else "SHORT"
    conviction = signal_to_noise(spread, mae)
    sign = 1 if signal == "LONG" else -1

    # P&L: only count if conviction is MEDIUM or HIGH
    pnl = sign * (real_base - ref_base) if conviction != "LOW" else None

    results.append({
        "week_start": str(test_df.index[0])[:10],
        "fc_base": fc_base,
        "ref_base": ref_base,
        "spread": round(spread, 2),
        "signal": signal,
        "conviction": conviction,
        "realized_base": real_base,
        "pnl": round(pnl, 2) if pnl is not None else None,
    })
    print(f"{test_df.index[0].date()}  spread={spread:+.1f}  {signal} ({conviction})  realized={real_base:.1f}  pnl={pnl}")

print("\nDone")

2026-04-03  spread=-49.9  SHORT (HIGH)  realized=51.8  pnl=49.099999999999994
2026-04-10  spread=+11.3  LONG (LOW)  realized=105.6  pnl=None
2026-04-17  spread=-11.2  SHORT (LOW)  realized=84.7  pnl=None
2026-04-24  spread=-32.9  SHORT (MEDIUM)  realized=57.4  pnl=28.35
2026-05-01  spread=+5.1  LONG (LOW)  realized=92.9  pnl=None
2026-05-08  spread=-0.8  SHORT (LOW)  realized=100.0  pnl=None
2026-05-15  spread=+18.6  LONG (MEDIUM)  realized=108.0  pnl=24.28
2026-05-22  spread=-14.2  SHORT (LOW)  realized=91.2  pnl=None
2026-05-29  spread=-6.5  SHORT (LOW)  realized=102.4  pnl=None
2026-06-05  spread=-6.9  SHORT (LOW)  realized=98.2  pnl=None

Done
